# 6. Tabelle e Figure Finali

Questo notebook non introduce nuovi modelli. Raccoglie gli output verificati dei notebook precedenti e produce tabelle/figure pronte per report e slide.


In [1]:
from pathlib import Path
import os

if Path.cwd().name != "notebook_final" and (Path.cwd() / "notebook_final").exists():
    os.chdir(Path.cwd() / "notebook_final")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "input" / "nasa_exoplanet_intelligence.csv"

import json
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

processed_dir = Path("data/processed")
cluster_dir = processed_dir / "clustering"
tables_dir = Path("reports/tables")
figures_dir = Path("reports/figures")
final_tables = Path("reports/final_tables")
final_figures = Path("reports/final_figures")
final_tables.mkdir(parents=True, exist_ok=True)
final_figures.mkdir(parents=True, exist_ok=True)

with open(cluster_dir / "clustering_metadata.json", encoding="utf-8") as f:
    cluster_meta = json.load(f)
with open(tables_dir / "supervised_summary.json", encoding="utf-8") as f:
    supervised_summary = json.load(f)

print("Clustering finale:")
print(cluster_meta)
print("\nSupervised summary keys:")
print(supervised_summary.keys())


Clustering finale:
{'highest_silhouette_raw': {'feature_set': 'B_planetary_orbital_stellar', 'method': 'agglomerative_single', 'k': 4, 'silhouette': 0.8228098598643738, 'cluster_sizes': '6145|2|2|1', 'min_cluster_size': 1, 'max_cluster_fraction': 0.9991869918699187, 'degenerate_heuristic': True}, 'best_by_balance_heuristic': {'feature_set': 'A_planetary_orbital', 'method': 'kmeans', 'k': 4, 'silhouette': 0.4470858036933995, 'cluster_sizes': '4054|1174|850|72', 'min_cluster_size': 72, 'max_cluster_fraction': 0.6591869918699187, 'degenerate_heuristic': False}, 'parsimonious_k2_alternative': {'feature_set': 'A_planetary_orbital', 'method': 'kmeans', 'k': 2, 'silhouette': 0.44599009086717084, 'cluster_sizes': '4196|1954', 'min_cluster_size': 1954, 'max_cluster_fraction': 0.6822764227642276, 'degenerate_heuristic': False}, 'silhouette_delta_k4_minus_k2': 0.0010957128262286675, 'final_feature_set': 'A_planetary_orbital', 'final_features': ['equilibrium_temp_k', 'orbital_eccentricity', 'orbit

## 6.1 Copia tabelle finali


In [2]:
table_sources = {
    "table_clustering_results.csv": cluster_dir / "clustering_comparison.csv",
    "table_cluster_summary.csv": tables_dir / "cluster_summary.csv",
    "table_cluster_means_raw.csv": tables_dir / "cluster_means_raw.csv",
    "table_cluster_means_zscore.csv": tables_dir / "cluster_means_zscore.csv",
    "table_planet_type_by_cluster.csv": tables_dir / "planet_type_by_cluster.csv",
    "table_star_type_by_cluster.csv": tables_dir / "star_type_by_cluster.csv",
    "table_habitable_zone_by_cluster.csv": tables_dir / "habitable_zone_flag_by_cluster.csv",
    "table_supervised_results_all.csv": tables_dir / "supervised_results_all.csv",
    "table_supervised_ablation_comparison.csv": tables_dir / "supervised_ablation_comparison.csv",
    "table_classification_report_best_B.csv": tables_dir / "classification_report_best_B.csv",
    "table_repeated_cv_macro_f1_setup_B.csv": tables_dir / "repeated_cv_macro_f1_setup_B.csv",
    "table_corrected_resampled_ttest_setup_B.csv": tables_dir / "corrected_resampled_ttest_setup_B.csv",
    "table_confidence_intervals_macro_f1.csv": tables_dir / "confidence_intervals_macro_f1.csv",
}

for out_name, src in table_sources.items():
    if src.exists():
        shutil.copyfile(src, final_tables / out_name)

summary_text = {
    "clustering": {
        "final_feature_set": cluster_meta["final_feature_set"],
        "final_method": cluster_meta["final_method"],
        "final_k": cluster_meta["final_k"],
        "final_silhouette": round(cluster_meta["final_silhouette"], 4),
        "bootstrap_mean": round(cluster_meta["bootstrap_mean"], 4),
        "bootstrap_std": round(cluster_meta["bootstrap_std"], 4),
    },
    "supervised": {
        "best_A": supervised_summary["best_A_by_test_macro_f1"],
        "best_B": supervised_summary["best_B_by_test_macro_f1"],
        "removed_features_setup_B": supervised_summary["ablation_removed_features"],
    },
}
with open(final_tables / "final_summary_text.json", "w", encoding="utf-8") as f:
    json.dump(summary_text, f, indent=4, ensure_ascii=False)

print("Tabelle finali:")
for p in sorted(final_tables.glob("*")):
    print("-", p.name)


Tabelle finali:
- final_summary_text.json
- table_classification_report_best_B.csv
- table_cluster2_data_quality_audit.csv
- table_cluster2_sensitivity_analysis.csv
- table_cluster_external_validation_ari.csv
- table_cluster_means_raw.csv
- table_cluster_means_zscore.csv
- table_cluster_summary.csv
- table_clustering_results.csv
- table_confidence_intervals_macro_f1.csv
- table_corrected_resampled_ttest_setup_B.csv
- table_habitable_zone_by_cluster.csv
- table_planet_type_by_cluster.csv
- table_repeated_cv_macro_f1_setup_B.csv
- table_star_type_by_cluster.csv
- table_supervised_ablation_comparison.csv
- table_supervised_results_all.csv


In [3]:
additional_cluster_tables = {
    "table_cluster_external_validation_ari.csv": tables_dir / "cluster_external_validation_ari.csv",
    "table_cluster2_data_quality_audit.csv": tables_dir / "cluster2_data_quality_audit.csv",
    "table_cluster2_sensitivity_analysis.csv": tables_dir / "cluster2_sensitivity_analysis.csv",
}
for output_name, source_path in additional_cluster_tables.items():
    if source_path.exists():
        shutil.copyfile(source_path, final_tables / output_name)
        print(f"Tabella aggiunta agli output finali: {output_name}")


Tabella aggiunta agli output finali: table_cluster_external_validation_ari.csv
Tabella aggiunta agli output finali: table_cluster2_data_quality_audit.csv
Tabella aggiunta agli output finali: table_cluster2_sensitivity_analysis.csv


## 6.2 Figure finali


In [4]:
clustering_results = pd.read_csv(cluster_dir / "clustering_comparison.csv")
cluster_summary = pd.read_csv(tables_dir / "cluster_summary.csv")
cluster_z = pd.read_csv(tables_dir / "cluster_means_zscore.csv", index_col=0)
cluster_name_map = dict(zip(cluster_summary["cluster"].astype(str), cluster_summary["cluster_name"]))
cluster_z.index = [cluster_name_map.get(str(idx), str(idx)) for idx in cluster_z.index]
cluster_z = cluster_z.rename(columns={"equilibrium_temp_k": "catalogued_temp_k"})
supervised_results = pd.read_csv(tables_dir / "supervised_results_all.csv")
ablation = pd.read_csv(tables_dir / "supervised_ablation_comparison.csv", index_col=0)
X_clustered = pd.read_csv(cluster_dir / "X_clustered.csv")
df_master = pd.read_csv(processed_dir / "df_master.csv")

plt.figure(figsize=(12, 6))
sns.barplot(
    data=clustering_results.sort_values("silhouette", ascending=False).head(12),
    x="silhouette",
    y="feature_set",
    hue="method",
)
plt.title("Top configurazioni clustering per silhouette")
plt.tight_layout()
plt.savefig(final_figures / "fig_silhouette_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))
for name, sub in clustering_results[clustering_results["method"] == "kmeans"].groupby("feature_set"):
    sub = sub.sort_values("k")
    plt.plot(sub["k"], sub["silhouette"], marker="o", label=name)
plt.title("KMeans silhouette by k")
plt.xlabel("k")
plt.ylabel("silhouette")
plt.legend()
plt.tight_layout()
plt.savefig(final_figures / "fig_kmeans_silhouette_by_k.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(max(10, cluster_z.shape[1] * 1.1), 2 + cluster_z.shape[0] * 0.75))
sns.heatmap(cluster_z, annot=True, cmap="coolwarm", center=0, fmt=".2f", linewidths=0.5)
plt.title("Cluster profile heatmap")
plt.tight_layout()
plt.savefig(final_figures / "fig_cluster_profile_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
x_col = "cluster_name" if "cluster_name" in cluster_summary.columns else "cluster"
sns.barplot(data=cluster_summary, x=x_col, y="size", palette="tab10")
plt.xticks(rotation=20, ha="right")
plt.title("Cluster sizes")
plt.tight_layout()
plt.savefig(final_figures / "fig_cluster_sizes.png", dpi=300, bbox_inches="tight")
plt.show()

X_final = X_clustered.drop(columns=["cluster"])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_final)
pca_df = pd.DataFrame({"PC1": coords[:, 0], "PC2": coords[:, 1], "cluster": X_clustered["cluster"]})
if "cluster_name" in cluster_summary.columns:
    name_map = dict(zip(cluster_summary["cluster"], cluster_summary["cluster_name"]))
    pca_df["cluster_label"] = pca_df["cluster"].map(name_map)
else:
    pca_df["cluster_label"] = pca_df["cluster"]
plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="cluster_label", palette="tab10", alpha=0.6, s=20)
plt.title(f"PCA clusters - explained variance {pca.explained_variance_ratio_.sum():.1%}")
plt.tight_layout()
plt.savefig(final_figures / "fig_pca_clusters.png", dpi=300, bbox_inches="tight")
plt.show()

for src_name, out_name, cmap in [
    ("planet_type_by_cluster.csv", "fig_planet_type_by_cluster.png", "Blues"),
    ("star_type_by_cluster.csv", "fig_star_type_by_cluster.png", "Greens"),
    ("habitable_zone_flag_by_cluster.csv", "fig_hz_by_cluster.png", "Oranges"),
]:
    p = tables_dir / src_name
    if p.exists():
        tab = pd.read_csv(p, index_col=0)
        plt.figure(figsize=(max(8, tab.shape[1] * 1.25), 2 + tab.shape[0] * 0.65))
        sns.heatmap(tab, annot=True, cmap=cmap, fmt=".2f", linewidths=0.5)
        plt.title(src_name.replace("_", " ").replace(".csv", ""))
        plt.tight_layout()
        plt.savefig(final_figures / out_name, dpi=300, bbox_inches="tight")
        plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(data=supervised_results, x="model", y="test_macro_f1", hue="setup")
plt.ylim(0, 1.05)
plt.title("Supervised test Macro-F1")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(final_figures / "fig_supervised_macro_f1.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
abl = ablation.reset_index().rename(columns={"index": "model"})
sns.barplot(data=abl.sort_values("delta_A_minus_B", ascending=False), x="delta_A_minus_B", y="model", color="steelblue")
plt.title("Macro-F1 drop after leakage-aware ablation")
plt.tight_layout()
plt.savefig(final_figures / "fig_ablation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

extra_figures = {
    "confidence_intervals_macro_f1.png": "fig_confidence_intervals_macro_f1.png",
    "confusion_matrix_best_B.png": "fig_confusion_matrix_best_B.png",
}
for src_name, out_name in extra_figures.items():
    src = figures_dir / src_name
    if src.exists():
        shutil.copyfile(src, final_figures / out_name)

print("Figure finali:")
for p in sorted(final_figures.glob("*.png")):
    print("-", p.name)


Figure finali:
- fig_ablation_comparison.png
- fig_cluster_profile_heatmap.png
- fig_cluster_sizes.png
- fig_confidence_intervals_macro_f1.png
- fig_confusion_matrix_best_B.png
- fig_hz_by_cluster.png
- fig_kmeans_silhouette_by_k.png
- fig_pca_clusters.png
- fig_planet_type_by_cluster.png
- fig_silhouette_comparison.png
- fig_star_type_by_cluster.png
- fig_supervised_macro_f1.png


## 6.2.1 Raw silhouette and complete K-Means comparison


In [5]:
plot_results = clustering_results.copy()
set_labels = {
    "A_planetary_orbital": "Set A",
    "B_planetary_orbital_stellar": "Set B",
}
method_labels = {
    "kmeans": "K-Means",
    "agglomerative_single": "Single linkage",
    "agglomerative_complete": "Complete linkage",
}
plot_results["configuration"] = (
    plot_results["feature_set"].map(set_labels)
    + " | "
    + plot_results["method"].map(method_labels)
    + " | k="
    + plot_results["k"].astype(str)
)

raw_top = plot_results.sort_values("silhouette", ascending=False).head(8)
kmeans_comparison = plot_results[plot_results["method"] == "kmeans"].sort_values(
    ["feature_set", "k"]
)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
sns.barplot(data=raw_top, x="silhouette", y="configuration", color="indianred", ax=axes[0])
axes[0].set_title("Raw silhouette: hierarchical maxima can be severely imbalanced")
axes[0].set_xlim(0, 0.9)

sns.barplot(data=kmeans_comparison, x="silhouette", y="configuration", color="steelblue", ax=axes[1])
axes[1].set_title("Complete K-Means comparison: both feature sets, k=2..8")
axes[1].set_xlim(0, 0.9)

for ax in axes:
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(final_figures / "fig_silhouette_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


## 6.3 Riepilogo finale


In [6]:
summary = json.loads((final_tables / "final_summary_text.json").read_text(encoding="utf-8"))
display(summary)


{'clustering': {'final_feature_set': 'A_planetary_orbital',
  'final_method': 'kmeans',
  'final_k': 4,
  'final_silhouette': 0.4471,
  'bootstrap_mean': 0.4116,
  'bootstrap_std': 0.0648},
 'supervised': {'best_A': {'setup': 'A_full',
   'model': 'Decision Tree',
   'cv_accuracy_mean': 0.9995901639344262,
   'cv_accuracy_std': 0.0005019446194227761,
   'cv_f1_macro_mean': 0.9991648851688435,
   'cv_f1_macro_std': 0.0011137734048558647,
   'test_accuracy': 1.0,
   'test_macro_f1': 1.0,
   'best_params': {'model__max_depth': None, 'model__min_samples_leaf': 1}},
  'best_B': {'setup': 'B_ablation',
   'model': 'Random Forest',
   'cv_accuracy_mean': 0.6245901639344262,
   'cv_accuracy_std': 0.019937166145828345,
   'cv_f1_macro_mean': 0.5265108733770891,
   'cv_f1_macro_std': 0.020074587751126267,
   'test_accuracy': 0.5959016393442623,
   'test_macro_f1': 0.49035381938980943,
   'best_params': {'model__max_depth': None,
    'model__min_samples_leaf': 5,
    'model__n_estimators': 200}},